# Tutorial 8: Noise Models and Error Mitigation

Running quantum finance algorithms on real hardware introduces noise.
This notebook demonstrates qufin's noise models and error mitigation strategies.

**Strategies covered**: ZNE, TREX, readout calibration, PEC, CDR.

**Reference**: Temme, Bravyi, Gambetta (2017). Czarnik et al. (2021).

In [ ]:
import numpy as np
np.random.seed(42)

## 1. Ideal vs Noisy Simulation

qufin provides 4 device noise profiles modeling real IBM hardware.

In [ ]:
from qufin.backends.qiskit_backend import QiskitAerBackend
from qufin.backends.noise_models import NoisyAerBackend, NoiseProfile

ideal_backend = QiskitAerBackend(shots=8192)

# Noisy backend with IBM Eagle r3 device profile
noisy_backend = NoisyAerBackend(
    profile=NoiseProfile.EAGLE_R3,
    shots=8192,
)

print(f"Ideal backend: {ideal_backend.backend_id}")
print(f"Noisy backend: {noisy_backend.backend_id}")
print(f"Noise profile: {noisy_backend.noise_profile}")

## 2. Noise Impact on a Simple Circuit

In [ ]:
from qiskit.circuit import QuantumCircuit

# Bell state circuit
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

ideal_counts = ideal_backend.run(qc, shots=8192).counts
noisy_counts = noisy_backend.run(qc, shots=8192).counts

print(f"Ideal counts: {ideal_counts}")
print(f"Noisy counts: {noisy_counts}")

## 3. Readout Error Calibration

In [ ]:
from qufin.backends.error_mitigation import calibrate_readout, mitigate_readout

# Calibrate the readout error (measures all basis states)
cal_matrix = calibrate_readout(noisy_backend, n_qubits=2, shots=8192)

print(f"Calibration matrix shape: {cal_matrix.shape}")
print(f"Diagonal (correct readout probs): {np.diag(cal_matrix).round(4)}")

In [ ]:
# Apply readout mitigation
mitigated = mitigate_readout(noisy_counts, cal_matrix)

print(f"Raw noisy:  {noisy_counts}")
print(f"Mitigated:  {mitigated}")
print(f"Ideal:      {ideal_counts}")

## 4. Zero-Noise Extrapolation (ZNE)

ZNE runs circuits at increasing noise levels and extrapolates to the zero-noise limit.

In [ ]:
from qufin.backends.error_mitigation import zne_extrapolate

# ZNE with scale factors [1, 2, 3]
zne_result = zne_extrapolate(
    circuit=qc,
    backend=noisy_backend,
    scale_factors=[1, 2, 3],
    shots=8192,
)

print(f"ZNE extrapolated value:  {zne_result.mitigated_value:.6f}")
print(f"Raw noisy value:         {zne_result.raw_value:.6f}")
print(f"Improvement:             {abs(zne_result.improvement):.4f}")

## 5. TREX (Twirled Readout Error eXtinction)

In [ ]:
from qufin.backends.error_mitigation import trex_mitigate

trex_result = trex_mitigate(
    circuit=qc,
    backend=noisy_backend,
    n_randomizations=32,
    shots=8192,
)

print(f"TREX mitigated value: {trex_result.mitigated_value:.6f}")
print(f"Raw noisy value:      {trex_result.raw_value:.6f}")

## 6. Noise Sweep

Compare results across different noise profiles.

In [ ]:
from qufin.backends.noise_models import sweep_noise

profiles = [NoiseProfile.EAGLE_R3, NoiseProfile.HERON_R1]

for profile in profiles:
    nb = NoisyAerBackend(profile=profile, shots=8192)
    counts = nb.run(qc, shots=8192).counts
    # Fidelity: fraction in expected states
    good = counts.get("00", 0) + counts.get("11", 0)
    total = sum(counts.values())
    print(f"  {str(profile):<25s} fidelity={good/total:.4f}")

## 7. Dynamical Decoupling

In [ ]:
from qufin.backends.dynamical_decoupling import apply_dd

# Apply XY4 dynamical decoupling sequence
qc_dd = apply_dd(qc, sequence="xy4")

dd_counts = noisy_backend.run(qc_dd, shots=8192).counts
good_dd = dd_counts.get("00", 0) + dd_counts.get("11", 0)
total_dd = sum(dd_counts.values())

print(f"Without DD fidelity: {good/total:.4f}")
print(f"With DD fidelity:    {good_dd/total_dd:.4f}")

## Summary

| Strategy | Overhead | Bias | Best For |
|:---------|:---------|:-----|:---------|
| Readout cal. | Low | Readout only | All circuits |
| ZNE | 3x circuit executions | Gate + readout | Short circuits |
| TREX | 32x randomizations | Readout | High readout error |
| PEC | Exponential sampling | Unbiased | Research |
| CDR | Training circuits | Gate errors | Variational |

**Next**: Tutorial 09 covers running on real IBM Quantum hardware.